# IMPORT NECCESSORY LIBRARIES

In [72]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# READ THE DATA

In [73]:
data=pd.read_csv('/content/tweets.csv')

In [59]:
data.head()

,id,label,tweet
0,1,0,#fingerprint #Pregnancy Test https://goo.gl/h1...
1,2,0,Finally a transparant silicon case ^^ Thanks t...
2,3,0,We love this! Would you go? #talk #makememorie...
3,4,0,I'm wired I know I'm George I was made that wa...
4,5,1,What amazing service! Apple won't even talk to...


## Data preprocessing


Clean and preprocess the 'tweet' column by removing noise such as mentions, hashtags, URLs, and special characters.



Function to clean the tweet text and apply it to the 'tweet' column.



In [60]:
import re        #Imports the regular expression module for pattern-based text manipulation.

def clean_tweet(tweet):
    # Remove mentions
    tweet = re.sub(r'@\w+', '', tweet)
    # Remove hashtags
    tweet = re.sub(r'#\w+', '', tweet)
    # Remove URLs
    tweet = re.sub(r'http\S+|www\S+', '', tweet)
    # Remove special characters, punctuation, and numbers
    tweet = re.sub(r'[^A-Za-z\s]+', '', tweet)
    # Convert to lowercase
    tweet = tweet.lower()
    # Remove leading and trailing whitespace
    tweet = tweet.strip()
    return tweet

data['cleaned_tweet'] = data['tweet'].apply(clean_tweet)
display(data[['tweet', 'cleaned_tweet']].head())

,tweet,cleaned_tweet
0,#fingerprint #Pregnancy Test https://goo.gl/h1...,test
1,Finally a transparant silicon case ^^ Thanks t...,finally a transparant silicon case thanks to ...
2,We love this! Would you go? #talk #makememorie...,we love this would you go
3,I'm wired I know I'm George I was made that wa...,im wired i know im george i was made that way
4,What amazing service! Apple won't even talk to...,what amazing service apple wont even talk to m...


## Tokenization and lemmatization


Tokenize the preprocessed tweets and perform lemmatization to reduce words to their base form.



Import necessary libraries for tokenization and lemmatization and define a function to tokenize and lemmatize tweets.



In [61]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
#function to lemmatize the tweet

def lemmatize_tweet(tweet):
    tokens = word_tokenize(tweet)
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return lemmatized_tokens

data['lemmatized_tweet'] = data['cleaned_tweet'].apply(lemmatize_tweet)


Display the head of the DataFrame to show the original and lemmatized tweets to verify the process.



In [62]:
display(data[['cleaned_tweet', 'lemmatized_tweet']].head())

,cleaned_tweet,lemmatized_tweet
0,test,[test]
1,finally a transparant silicon case thanks to ...,"[finally, a, transparant, silicon, case, thank..."
2,we love this would you go,"[we, love, this, would, you, go]"
3,im wired i know im george i was made that way,"[im, wired, i, know, im, george, i, wa, made, ..."
4,what amazing service apple wont even talk to m...,"[what, amazing, service, apple, wont, even, ta..."


## Feature extraction


Convert the text data into numerical features using techniques like TF-IDF.


Import the TfidfVectorizer, instantiate it, join the lemmatized tokens into strings, and transform the text data into a TF-IDF matrix.



In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Join the lemmatized tokens back into strings
data['lemmatized_tweet_str'] = data['lemmatized_tweet'].apply(lambda x: ' '.join(x))

# Instantiate TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Example: limit features to 5000

# Fit and transform the text data
tfidf_matrix = tfidf_vectorizer.fit_transform(data['lemmatized_tweet_str'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (7920, 5000)


## Model training


Train a sentiment analysis model  on the labeled data.



Train a sentiment analysis model using the TF-IDF features and the provided labels. This involves splitting the data, initializing a model, and fitting it to the training data.



In [64]:

X = tfidf_matrix
y = data['label']# 0==> MEANS POSITIVE , 1 ==> MEANS NEGATIVE

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#model = MultinomialNB()
#model=LogisticRegression()

model = LinearSVC()


# Train the model
model.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


## Model evaluation

Evaluate the performance of the trained model using appropriate metrics.


Import the necessary metrics from sklearn.metrics, make predictions on the test data using the trained model, and calculate and print the evaluation metrics.



In [65]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Make predictions on the test data
y_pred = model.predict(X_test)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)


# Print the metrics
print(f"Accuracy: {accuracy:.4f}")


Accuracy: 0.8554


## Sentiment Prediction


Use the trained model to predict sentiment on new, unseen tweets.


Define a list of new tweets, preprocess them using the same steps as the training data, transform them using the fitted TF-IDF vectorizer, and use the trained model to predict the sentiment.

In [71]:
# test on  new tweets
new_tweets = [
    "This is a great product! I love it.",
    "I am very disappointed with the service.",
    "Neutral tweet about the weather today.",
    "Amazing experience, highly recommended!",
    "Terrible quality, a complete waste of money."
]


# Preprocess the new tweets using the same steps as the training data
cleaned_new_tweets = [clean_tweet(tweet) for tweet in new_tweets]
lemmatized_new_tweets = [' '.join(lemmatize_tweet(tweet)) for tweet in cleaned_new_tweets]

# Transform the new tweets using the fitted TF-IDF vectorizer
X_new = tfidf_vectorizer.transform(lemmatized_new_tweets)

# Predict the sentiment of the new tweets
predictions = model.predict(X_new)

# Print the predictions
sentiment_map = {0: "positive", 1: "negative"}
for tweet, prediction in zip(new_tweets, predictions):
    print(f"Tweet: {tweet} -> Predicted Sentiment: {sentiment_map[prediction]}")

Tweet: This is a great product! I love it. -> Predicted Sentiment: positive
Tweet: I am very disappointed with the service. -> Predicted Sentiment: negative
Tweet: Neutral tweet about the weather today. -> Predicted Sentiment: positive
Tweet: Amazing experience, highly recommended! -> Predicted Sentiment: positive
Tweet: Terrible quality, a complete waste of money. -> Predicted Sentiment: negative


THE MODEL PREDICTED THE SENTIMENT CORRECTLY ON THE UNSEEN TEST DATA.